# Hi-EF source-folder-held-out split

This notebook loads the frozen split from the `experiments` branch and verifies its leakage invariants before training. The numerical folder prefix is treated as an anonymized source unit, likely an episode-level proxy pending author confirmation.

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

REPO_DIR = Path('/kaggle/working/hi-ef-materials')
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'experiments',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO_DIR)
    ], check=True)
print('Repository:', REPO_DIR)

In [ ]:
subprocess.run([
    'python', str(REPO_DIR / 'experiments/build_source_folder_split.py'),
    '--sample-csv', str(REPO_DIR / 'sample.csv'),
    '--annotation-csv', str(REPO_DIR / 'annotation.csv'),
    '--output-dir', str(REPO_DIR / 'experiments/manifests'),
    '--seed', '42'
], check=True)

In [ ]:
manifest_path = REPO_DIR / 'experiments/manifests/source_folder_split_seed42.csv'
audit_path = REPO_DIR / 'experiments/manifests/source_folder_split_seed42_audit.json'
manifest = pd.read_csv(manifest_path, dtype={'source_folder': str})
audit = json.loads(audit_path.read_text())
assert audit['passed'], audit['pairwise_leakage']
assert manifest.groupby('source_folder')['split'].nunique().max() == 1
display(manifest.groupby('split').agg(samples=('sample_id', 'size'), source_folders=('source_folder', 'nunique')))
display(pd.crosstab(manifest['clip4_emotion'], manifest['split']))
audit['pairwise_leakage']

In [ ]:
train_rows = manifest[manifest['split'] == 'train'].reset_index(drop=True)
val_rows = manifest[manifest['split'] == 'val'].reset_index(drop=True)
test_rows = manifest[manifest['split'] == 'test'].reset_index(drop=True)
print(len(train_rows), len(val_rows), len(test_rows))

## Training hand-off

Use `train_rows` and `val_rows` for development and checkpoint selection. Do not inspect `test_rows` metrics until the model configuration and seeds are fixed.